# PySpark DataFrame Practice Notebook

In this notebook, we will learn:

- How to read data into a DataFrame
- Basic DataFrame operations
- Filtering and transformations
- Aggregations
- Joins
- Window functions
- Writing data

This simulates a real-world data engineering workflow.

## Step 1: Initialize Spark Session

IN databricks we do not need to create SparkSession as the server here already create that.

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
    .appName("Pyspark Practice")\
    .getOrCreate()

spark

## Step 2: Read Data into DataFrame

We will load a file into a PySpark DataFrame.

In [0]:
df = spark.read.table("ecommerce_analytics.data_warehouse.fact_order_items")

In [0]:
df.display()

## Step 3: Explore Data

Let’s understand the structure and schema.

In [0]:
df.printSchema()

In [0]:
df.columns

In [0]:
df.describe().display()

In [0]:
df.count()

## Step 4: Select Columns

In [0]:
df.select("customer_id","order_date_key","product_id","price").display()


## Step 5: Filter Data

In [0]:
df.where("customer_id == 4401099")

df.filter((df["customer_id"]== 4401099) & (df["order_date_key"] == "20190802")).display()

## Step 6: Add or Modify Columns

In [0]:
from pyspark.sql.functions import col 
df = df.withColumn("profit", col("amount") - col("price"))

In [0]:
df.display()

## Step 7: Rename Columns

In [0]:
df = df.withColumnRenamed("profit", "PROFIT")
df.display()

## Step 8: Drop Columns

In [0]:
df.drop("PROFIT")

## Step 9: Handle Missing Values

In [0]:
df.fillna({'promo_id': 0}).display()

In [0]:
from pyspark.sql.functions import lit
df.fillna({'promo_id': "NA"}).display()


In [0]:
# Deleted null values rows
df.dropna().display()

## Step 10: Remove Duplicate Records

In [0]:
df_unique = df.dropDuplicates()

In [0]:
# if want to see duplicates only
df.exceptAll(df_unique).display()

In [0]:
df.distinct().count()

## Step 11: Aggregations

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import sum

df.groupBy("customer_id").agg(
    sum("amount").alias("total_sales")
).orderBy(col("total_sales").desc()).display()

In [0]:
from pyspark.sql.functions import sum

cust_sale = df.groupBy("customer_id").agg(
    sum("amount").alias("total_sales")
).orderBy("total_sales", ascending= False).display()

In [0]:
from pyspark.sql import functions as F

cust_sales = df.groupBy("customer_id").agg(
    F.sum("amount").alias("total_sales"),
    F.count("customer_id").alias("no_of_purchases"),
    F.avg("amount").alias("avg_sales")
).orderBy("total_sales", ascending= False)

## Step 12: Sorting Data

## Step 13: Joins

We will join two DataFrames.

Task -> Customer Name should be all Small letter and without commas(f and l name)

In [0]:
df_cust = spark.read.table("ecommerce_analytics.data_warehouse.dim_customer")

In [0]:
cust_sales.join(df_cust, "customer_id", "left")\
    .select("customer_name", "no_of_purchases", "total_sales", "avg_sales")\
    .display()

In [0]:
# Cleaning customer name
from pyspark.sql.functions import col, initcap,regexp_replace, trim

df_clean_cust_name = cust_sales.join(df_cust, "customer_id", "left")\
    .withColumn(
        "customer_name_clean",
        trim(       #removes extra spaces
            initcap(      #convert to proper case Smith John
                regexp_replace(
                    col("customer_name"),   #REmove comma + spaces
                    ",\\s*",
                    " ")
            )
        )
    ) \
    .select("customer_name_clean", "no_of_purchases", "total_sales", "avg_sales")

df_clean_cust_name.display()


## Step 14: Window Functions

In [0]:
from pyspark.sql.window import Window
w = Window.partitionBy("customer_id").orderBy(F.desc("amount"))
df.withColumn("row_number", F.row_number().over(w)).where("row_number == 1").display()

## Step 15: Conditional Columns

## Step 16: Date Functions

## Step 17: Cache DataFrame

## Step 18: Write Data to Storage

## Step 19: Using Spark SQL

## Key Learnings

- PySpark is lazy → transformations are not executed until an action
- Always use column functions instead of Python logic
- Prefer DataFrame API over RDDs
- Optimize using:
  - partitioning
  - caching
  - predicate pushdown